Participantes:

Carlos Augusto Freire Maia de Oliveira - RA: 21.00781-0

Cesar Augusto Bresciani Junior - RA: 21.00478-0

Enzo Leonardo Sabatelli de Moura - RA: 21.01535-0

# Classificador de texto Humano x IA

O projeto busca treinar um modelo capaz de identificar se um texto foi escrito por um humano ou por uma IA.

### Dados utilizados

Foram criados datasets baseados em três consultas:

* Artigos da Wikipédia
* Reviews de produtos do Mercado Livre
* Redações nota 1000 do Enem

Para cada dataset foram criados prompts para gerar dados equivalentes.

Modelos utilizados para geração de dataset de IA:
* Wikipédia:
    * gemini-1.5-flash,
    * gemini-1.5-flash-8b,
    * gemini-1.5-pro,
    * gemini-pro
* Redação:
    * gemini-2.5-flash
* Review:
    * gemini-2.5-pro

### Criação do modelo

Foram criados dois modelos para realizar a classificação de texto:
* LinearSVC (TF-IDF)
* BERTimbau (Modelo BERT pré-treinado com textos da língua portuguesa)

In [ ]:
# Imports gerais
import pandas as pd
import numpy as np

___
## Dados Wikipédia

### Coleta de dados humanos

Foi utilizada a api da Wikipédia para a coleta dos artigos. Foi utilizado o seguinte formato:



```python
WIKI_API_URL = "https://{lang}.wikipedia.org/w/api.php"
HEADERS = {"User-Agent": "WikipediaDataCollector/1.0 (+https://example.com/contact)"}

# Request
def wiki_request(params: dict, lang: str, retries: int = 3, backoff: float = 1.5) -> dict:
    url = WIKI_API_URL.format(lang=lang)
    last_err = None
    for i in range(retries):
        try:
            resp = requests.get(url, params=params, headers=HEADERS, timeout=20)
            if resp.status_code == 403:
                time.sleep(backoff ** (i + 1))
                continue
            resp.raise_for_status()
            return resp.json()
        except Exception as e:
            last_err = e
            time.sleep(backoff ** (i + 1))
    raise RuntimeError(f"Falha na Wikipedia após {retries} tentativas: {last_err}")
```

A lista de temas de artigos utilizados está presente em [teste.json](https://github.com/Becker23/projeto-cd-ia/blob/main/teste.json)

O código completo da coleta está em [scrap_chat_wiki.py](https://github.com/Becker23/projeto-cd-ia/blob/main/scrap_chat_wiki.py)


### Geração de artigos com IA

Para a geração de artigos foi utilizado o seguinte prompt:

```python
prompt = (
    "Reescreva o texto abaixo de forma objetiva, neutra e enciclopédica,\n"
    "sem incluir títulos ou seções, sem linguagem promocional,\n"
    "sem estrutura de artigo didático.\n"
    "Estilo desejado: semelhante ao da Wikipédia.\n\n"
    "Texto de base:\n"
    f"{base_text}\n"
)
```



In [ ]:
# Head do dataset da Wikipédia

df_wiki = pd.read_json("https://raw.githubusercontent.com/Becker23/projeto-cd-ia/refs/heads/main/saida_wiki/set_wiki.json")
df_wiki.head()

,titulo,texto,classe,fonte_path
0,Aerodinâmica,"Aerodinâmica, do grego antigo ἀήρ aer (ar) + δ...",humano,C:\Users\Enzo\Documents\projects\projeto-cd-ia...
1,Aerodinâmica,"Aerodinâmica é o estudo do movimento do ar, es...",ia,C:\Users\Enzo\Documents\projects\projeto-cd-ia...
2,Algoritmo_genético,Um algoritmo genético (AG) é uma técnica de bu...,humano,C:\Users\Enzo\Documents\projects\projeto-cd-ia...
3,Algoritmo_genético,Um algoritmo genético (AG) é uma técnica de bu...,ia,C:\Users\Enzo\Documents\projects\projeto-cd-ia...
4,Aminoácido,"Aminoácidos são compostos de carbono (C), hidr...",humano,C:\Users\Enzo\Documents\projects\projeto-cd-ia...


---
## Dados de redação




### Coleta de dados humanos

Para as redações do enem, foram utilizadas as redações nota 1000 divulgadas pelo https://g1.globo.com/educacao

Foram utilizadas 71 redações nota 1000 do período de 2018 até 2024.

Foram gerados 71 textos de IA baseado nos temas de cada redação nesse mesmo período.

A coleta dos textos foi manual, utilizando o código [collect_human_texts.py](https://github.com/Becker23/projeto-cd-ia/blob/main/redacao/collect_human_texts.py) para auxiliar no processo.

In [ ]:
# Head do dataset de redação (humano)
df_redacao_humano = pd.read_parquet("https://github.com/Becker23/projeto-cd-ia/raw/refs/heads/main/redacao/redacao_2018.parquet")
df_redacao_humano.head()

,text,label
0,"No livro “1984” de George Orwell, é retratado ...",human
1,“Black Mirror” é uma série americana que retra...,human
2,É fato que a tecnologia revolucionou a vida em...,human
3,O mundo conheceu novos equipamentos ao longo d...,human
4,A série britânica “Black Mirror” é caracteriza...,human


### Geração de redação com IA

Para geração de redação por IA foi utilizado o seguinte modelo de prompt:
<details>
<summary>
Prompt
</summary>

```
## 🧠 **PROMPT — Guia Definitivo para Criação de Redação ENEM (com textos motivadores)**

> **Instrução para a IA:**
> Você é uma inteligência artificial especializada em **redações do ENEM**.
> Sua tarefa é **produzir uma redação dissertativo-argumentativa** seguindo **rigorosamente** as normas e critérios do **Exame Nacional do Ensino Médio (ENEM)**, utilizando o **tema** e os **textos motivadores** que serão fornecidos pelo usuário.
>
> ---
>
> ### 🧩 1. Estrutura obrigatória
>
> A redação deve ter **entre 20 e 30 linhas**, organizada em **prosa dissertativo-argumentativa**, com:
>
> * **Introdução:** Apresentação do tema e da tese (ponto de vista a ser defendido).
> * **Desenvolvimento 1:** Primeiro argumento, fundamentado logicamente.
> * **Desenvolvimento 2:** Segundo argumento, complementando o anterior.
> * **Conclusão:** Retomada da tese e apresentação de uma **proposta de intervenção social detalhada**.
>
> O texto deve ser **coeso, coerente, objetivo e formal**, sem listas, diálogos ou estrutura narrativa.
>
> ---
>
> ### 📚 2. Uso obrigatório dos textos motivadores
>
> * Os **textos motivadores** fornecidos devem ser **lidos, compreendidos e utilizados** para embasar os argumentos.
> * É **obrigatório** incorporar ideias, dados ou reflexões extraídos deles, **sem cópia literal**.
> * O uso deve ser **crítico, interpretativo e integrado** à argumentação, demonstrando repertório sociocultural.
>
> ---
>
> ### 🧱 3. Regras fundamentais
>
> * Utilize **apenas a norma culta da língua portuguesa**.
> * O texto deve ser **inteiramente original** e **respeitoso aos direitos humanos**.
> * **A tese** precisa ser clara e defendida ao longo de todo o texto.
> * Use **conectivos e articuladores** adequados para garantir coesão e progressão lógica.
> * **Jamais fuja do tema** ou altere o tipo textual.
>
> ---
>
> ### 🎯 4. Competências do ENEM
>
> 1. **Domínio da norma culta** da língua portuguesa.
> 2. **Compreensão do tema** e adequação ao gênero dissertativo-argumentativo.
> 3. **Organização de argumentos** de modo coerente e coeso.
> 4. **Uso apropriado de recursos linguísticos** na argumentação.
> 5. **Proposta de intervenção** detalhada, viável e ética, respeitando os direitos humanos.
>
> ---
>
> ### 🧩 5. Estrutura da proposta de intervenção
>
> A conclusão deve conter uma **proposta de intervenção** com os cinco elementos obrigatórios:
>
> * **Agente:** quem executa a ação;
> * **Ação:** o que será feito;
> * **Meio/modo:** como será feito;
> * **Finalidade:** por que será feito (objetivo social);
> * **Detalhamento:** local, recursos, etapas ou consequências positivas.
>
> ---
>
> ### ⚖️ 6. Postura ética e adequação
>
> * **Proibido:** discurso de ódio, ironia, linguagem informal, plágio ou fuga ao tema.
> * **Obrigatório:** tom formal, postura crítica e respeito aos direitos humanos.
>
> ---
>
> ### 🧠 7. Forma de resposta esperada
>
> Ao receber o **tema** e os **textos motivadores**, siga estes passos:
>
> 1. Analise o tema e identifique o problema central.
> 2. Utilize os textos motivadores para embasar ideias e dados.
> 3. Elabore uma **tese clara**.
> 4. Desenvolva **dois parágrafos argumentativos**.
> 5. Conclua com uma **proposta de intervenção detalhada**.
>
> ---
>
> ### 🗒️ 8. Forma final de saída
>
> 🔴 **Muito importante:**
>
> * Sua resposta deve conter **apenas e somente o texto da redação completa**,
> * **Sem título**, **sem comentários**, **sem explicações**, **sem marcações**, **sem repetições do tema** e **sem qualquer texto fora da redação**.
> * O texto deve começar imediatamente com a introdução e terminar com a conclusão.
>
> ---
>
> ### 🧾 9. Exemplo de uso
>
> Quando o usuário enviar algo como:
>
> ```
> Tema: A influência da tecnologia nas relações humanas.  
>
> Textos motivadores:  
> [Texto 1] Pesquisas apontam que a comunicação digital tem substituído o contato presencial em diversos contextos sociais.  
> [Texto 2] Estudos psicológicos destacam o aumento do isolamento social em jovens conectados à internet.  
> [Texto 3] O avanço tecnológico trouxe facilidades, mas também desafios para a empatia e o convívio interpessoal.
> ```
>
> Você deverá gerar **apenas a redação completa**, conforme todas as instruções acima.
>
> ---
>
> **Fim do guia.**
> Aguarde o envio do *tema* e dos *textos motivadores* antes de >redigir.
>---
>Tema: [TEMA]
>---
>Texto 1: [TEXTO1]
>---
[... mais textos motivadores]
```
</details>

O código utilizado para gerar estes textos foi [generate_ai_texts.py](https://github.com/Becker23/projeto-cd-ia/blob/main/redacao/generate_ai_texts.py)

In [ ]:
# Head do dataset de IA
df_redacao_ia = pd.read_parquet("https://github.com/Becker23/projeto-cd-ia/raw/refs/heads/main/redacao/ai_texts.parquet")
df_redacao_ia.head()

,ano_enem,response,label
0,2024,"A herança africana no Brasil, compreendida com...",ai
1,2024,"A herança africana no Brasil, vasta em saberes...",ai
2,2024,"A herança africana no Brasil, compreendida com...",ai
3,2024,A história do Brasil é marcada por uma profund...,ai
4,2024,"A herança africana no Brasil, rica em saberes,...",ai


---
## Dados de Reviews

### Coleta de dados humanos

A coleta de reviews humanas foi feita de maneira manual. Para cada produto do arquivo product_links.txt (disponível em ), foram coletadas 3 reviews. Para a coleta, foram considerados aspectos como tamanho do texto e escrita (textos com diversos erros de escrita não foram incluídos). Além disso, foram escolhidas reviews tanto boas quanto neutras ou ruins, de maneira aleatória.

In [ ]:
df_reviews_humano = pd.read_json('https://raw.githubusercontent.com/Becker23/projeto-cd-ia/refs/heads/main/reviews_ml/set_human.json')
df_reviews_humano.head()

,product_title,product_link,review_text
0,Panela De Pressão Fechamento Externo Antiadere...,https://www.mercadolivre.com.br/panela-de-pres...,"Gostei muito, estou fazendo feijão! Pega muito..."
1,Panela De Pressão Fechamento Externo Antiadere...,https://www.mercadolivre.com.br/panela-de-pres...,Otima panela ja usei e cozinha rapido os alime...
2,Panela De Pressão Fechamento Externo Antiadere...,https://www.mercadolivre.com.br/panela-de-pres...,"Me pareceu ser uma boa panela, mas ainda não u..."
3,Jogo De Lençol 400 Fios Ponto Palito Cama Quee...,https://produto.mercadolivre.com.br/MLB-396505...,Gostei muito! Pelo preço está excelente! Só a ...
4,Jogo De Lençol 400 Fios Ponto Palito Cama Quee...,https://produto.mercadolivre.com.br/MLB-396505...,A qualidade do tecido e boa o acabamento do po...


### Geração de reviews com IA

Já para a geração de reviews com IA foi utilizado o Gemini 2.5 Pro, com o seguinte prompt:

(Junto ao prompt foi enviado o arquivo product_links.txt)
<details>
<summary>
Prompt
</summary>

```
Este arquivo contém links para diversos produtos no Mercado Livre. Seu objetivo é analisar cada produto como um comprador, e gerar, de maneira aleatória, 3 reviews, podendo serem reviews boas, neutras ou ruins (não necessariamente uma de cada tipo). Limite-se a até 75 palavras para cada review.
```



In [ ]:
df_reviews_ia = pd.read_json('https://raw.githubusercontent.com/Becker23/projeto-cd-ia/refs/heads/main/reviews_ml/set_ai.json')
df_reviews_ia.head()

,product_title,product_link,review_text
0,Panela De Pressão Fechamento Externo Antiadere...,https://www.mercadolivre.com.br/panela-de-pres...,Amei a panela! Super segura com o fechamento e...
1,Panela De Pressão Fechamento Externo Antiadere...,https://www.mercadolivre.com.br/panela-de-pres...,"A panela é boa, cozinha rápido e a cor é bonit..."
2,Panela De Pressão Fechamento Externo Antiadere...,https://www.mercadolivre.com.br/panela-de-pres...,Que decepção! Usei a panela 3 vezes e a válvul...
3,Jogo De Lençol 400 Fios Ponto Palito Cama Queen,https://produto.mercadolivre.com.br/MLB-396505...,"Lençol maravilhoso, super macio e confortável...."
4,Jogo De Lençol 400 Fios Ponto Palito Cama Queen,https://produto.mercadolivre.com.br/MLB-396505...,Gostei bastante do jogo de lençol. O tecido é ...


Dataset completo:

In [ ]:
df = pd.read_json("https://github.com/Becker23/projeto-cd-ia/raw/refs/heads/main/dataset_final.json")
df.head()

,texto,classe
0,"Gostei muito, estou fazendo feijão! Pega muito...",humano
1,Otima panela ja usei e cozinha rapido os alime...,humano
2,"Me pareceu ser uma boa panela, mas ainda não u...",humano
3,Gostei muito! Pelo preço está excelente! Só a ...,humano
4,A qualidade do tecido e boa o acabamento do po...,humano


In [ ]:
print(len(df))

679


# Treinamento e testes dos modelos

## Treinamento de TF-IDF + LinearSVC

Foi utilizado TF-IDF (Term Frequency-Inverse Document Frequency) para vetorizar o dataset e então treinar uma SVM (Support Vector Machine) para classificação.

In [ ]:
# Imports utilizados

import os
import re
import glob
import json
import pickle
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


In [ ]:
df = pd.read_json("https://github.com/Becker23/projeto-cd-ia/raw/refs/heads/main/dataset_final.json")
df.head()

,texto,classe
0,"Gostei muito, estou fazendo feijão! Pega muito...",humano
1,Otima panela ja usei e cozinha rapido os alime...,humano
2,"Me pareceu ser uma boa panela, mas ainda não u...",humano
3,Gostei muito! Pelo preço está excelente! Só a ...,humano
4,A qualidade do tecido e boa o acabamento do po...,humano


Na construção do modelo, foram usados os seguintes parâmetros no vetorizador:
*   lowercase=True (default), para assegurar que palavras como "Texto" e "texto" sejam lidas igualmente
*   ngram_range=(1,2), que faz o vetorizador trabalhar tanto com unigramas quanto bigramas
*   max_df=0.9, que ignora palavras com frequência maior que 90% nos documentos, para evitar palavras com significado muito pequeno no modelo
*   min_df=1 (default), que faz com que todas as palavras que apareçam pelo menos uma vez nos documentos sejam consideradas na vetorização
*   max_features=20000, que considera apenas as 20000 palavras mais frequentes nos documentos

Já na construção do próprio modelo LinearSVC, foi usado C=1.0 (default). Essa constante é referente a "força de regularização" do modelo.

Quanto maior o valor, mais ajustado aos dados de treino o modelo será (menos regularização). Quanto menor o valor, mais simples o modelo será (mais regularização).

Essa constante foi mantida em 1.0 por ser um valor balanceado e padrão para todos os modelos.



Além disso, para separação dos dados em teste e treino foram usados os seguintes parâmetros:

*   test_size=0.3: 30% dos exemplos vão para o conjunto de teste, 70% para treino.

*   random_state=42: fixa a semente do gerador aleatório para que a divisão seja reprodutível.

*   stratify=df["classe"]: garante que a distribuição das classes no treino e no teste seja a mesma da distribuição original (importante se as classes são desbalanceadas).

In [ ]:
# Separando dados para treino
X_train, X_test, y_train, y_test = train_test_split(
        df["texto"], df["classe"], test_size=0.3, random_state=42, stratify=df["classe"]
    )

# Criando vectorizer

vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),  # unigrams + bigrams
    max_df=0.9,
    min_df=1,
    max_features=20000,
)

X_train_tfidf = vectorizer.fit_transform(X_train)

# Criando modelo LinearSVC
model_svc = LinearSVC(C=1.0)
model_svc.fit(X_train_tfidf, y_train)

# Transformando dados de teste usando vectorizer
X_test_tfidf = vectorizer.transform(X_test)

y_pred = model_svc.predict(X_test_tfidf)

# Metricas e reports
acc = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, digits=4)
cm = confusion_matrix(y_test, y_pred, labels=["humano", "ia"]).tolist()


In [ ]:
print("Resumo de treinamento simples (TF-IDF + LinearSVC):")
print(f"- Amostras treino: {len(X_train)} | teste: {len(X_test)}")
print(f"- Accuracy teste: {acc:.4f}")
print("Matriz de confusão [rows: humano, ia]:")
print(np.array(cm))
print("\nClassification report:\n", report)

Resumo de treinamento simples (TF-IDF + LinearSVC):
- Amostras treino: 475 | teste: 204
- Accuracy teste: 0.8137
Matriz de confusão [rows: humano, ia]:
[[81 20]
 [18 85]]

Classification report:
               precision    recall  f1-score   support

      humano     0.8182    0.8020    0.8100       101
          ia     0.8095    0.8252    0.8173       103

    accuracy                         0.8137       204
   macro avg     0.8139    0.8136    0.8137       204
weighted avg     0.8138    0.8137    0.8137       204



O modelo apresenta uma acurácia boa para os dados de teste. Porém, nós questionamos essa acurácia, pensando que as LLMs usariam palavras em uma frequência parecida com humanos. Ao testar na nossa aplicação que analisa textos, confirmamos nossa dúvida.

A questão é que a vetorização não é capaz de entender nuances e contextos de forma profunda. Por exemplo, a frase "O gato preto dorme sobre o tapete macio" seria igual à frase "Sobre o tapete macio dorme o gato preto" para o TF-IDF e consequentemente pelo LinearSVC, então nesta configuração não estão sendo analisados padrões de escrita.

Para mitigar isso, foram utilizados bigramas além dos unigramas para uma análise maior de contexto. Porém, mesmo assim, a análise de contexto não é profunda. Além disso, utilizar mais palavras na análise (como trigramas ou tetragramas) não alteram significativamente a acurácia do modelo e servem apenas para deixá-lo mais pesado e lento. (Isso é mostrado na parte de Cross-validation deste notebook)

Outro fator são formatações que podem estar mais presentes em comportamento de IA ou comportamento de humanos. Vimos que o modelo possui um viés em pensar que textos utilizando travessão (-) para aposto são feitos por IA.

## Fine-tuning do BERTimbau

O uso do BERTimbau nos auxilia de muitos modos. O modelo já está treinado para a língua portuguesa. Diferente do TF-IDF, ele não analisa apenas a frequência, mas o contexto também. Ele indentificará mais nuances e padrões nos textos.

Treinamos em cima do modelo bert-base-portuguese-cased ([Hugging Face](https://huggingface.co/neuralmind/bert-base-portuguese-cased)), realizando um transfer-learning com o nosso dataset.

Finalmente, faremos uso de Deep Learning para o treinamento do modelo.

Os seguintes parâmetros foram usados no treinamento:
*   MAX_LENGTH = 256, que limita o comprimento máximo de tokens passado ao modelo. Foi reduzido de 512 para poupar memória em tempo de execução, enquanto mantém contexto suficiente para os textos usados.
*   BATCH_SIZE = 4, que limita o número de elementos processados a cada passo de treinamento. Foi reduzida de 8, também para poupar memória.
*   EPOCHS = 5, que diz respeito ao número de passagens completas pelos dados de treino. 5 épocas foram suficientes para mitigar bastante a perda enquanto o tempo de treinamento continua satisfatório.
*   LEARNING_RATE = 2e-5, taxa de aprendizado para o otimizador durante o fine-tuning. 2e-5 é considerado um ponto de partida equilibrado para o modelo BERT. Caso seja maior, o modelo aprenderá mais rápido (em menos épocas), porém a perda de treinamento pode se tornar instável.



In [ ]:
# Imports

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
from pathlib import Path
import json

# Configurações de dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Constantes
MAX_LENGTH = 256
BATCH_SIZE = 4
EPOCHS = 5
LEARNING_RATE = 2e-5

In [ ]:
# Classe para datasets
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx])
        label = 1 if self.labels.iloc[idx] == "ia" else 0

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long),
        }

Para a separação dos conjuntos de teste e treino, foram usados os mesmos parâmetros do que no modelo linear.

Já neste modelo, foi usado num_labels=2, que configura a saída para previsão de duas classes ("humano" ou "ia"). Isso faz com que o modelo retorne logits de dimensão [batch_size, 2].

In [ ]:
# Carregar dataset
df = pd.read_json("https://github.com/Becker23/projeto-cd-ia/raw/refs/heads/main/dataset_final.json")

# Separar treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    df["texto"], df["classe"], test_size=0.3, random_state=42, stratify=df["classe"]
)

# Tokenizer e modelo BERTimbau
tokenizer = BertTokenizer.from_pretrained("neuralmind/bert-base-portuguese-cased")
model_bert = BertForSequenceClassification.from_pretrained(
    "neuralmind/bert-base-portuguese-cased", num_labels=2
).to(device)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

In [ ]:
# Criando dataset e dataloader
train_dataset = TextClassificationDataset(X_train, y_train, tokenizer, MAX_LENGTH)
test_dataset = TextClassificationDataset(X_test, y_test, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

# Optimizer
optimizer = AdamW(model_bert.parameters(), lr=LEARNING_RATE)


Finalmente, faremos uso de Deep Learning para o treinamento do modelo.

In [ ]:
print("Training BERT model...")

for epoch in range(EPOCHS):
    model_bert.train()
    total_loss = 0

    for batch_idx, batch in enumerate(train_loader):

        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model_bert(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

        # Liberar memória
        del outputs
        del loss
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")

Training BERT model...
Epoch 1/5, Loss: 0.5364
Epoch 2/5, Loss: 0.1741
Epoch 3/5, Loss: 0.0622
Epoch 4/5, Loss: 0.0454
Epoch 5/5, Loss: 0.0028


In [ ]:
# Avaliando o modelo
model_bert.eval()
predictions = []
actual_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]

        outputs = model_bert(input_ids=input_ids, attention_mask=attention_mask)
        _, preds = torch.max(outputs.logits, dim=1)

        predictions.extend(preds.cpu().tolist())
        actual_labels.extend(labels.cpu().tolist())

In [ ]:
# Exibindo resultados de métricas
label_map = {0: "humano", 1: "ia"}
pred_labels = [label_map[p] for p in predictions]
true_labels = [label_map[l] for l in actual_labels]

acc = accuracy_score(true_labels, pred_labels)
report = classification_report(true_labels, pred_labels, digits=4)
cm = confusion_matrix(true_labels, pred_labels, labels=["humano", "ia"]).tolist()

metrics = {
    "accuracy": float(acc),
    "labels": ["humano", "ia"],
    "confusion_matrix": cm,
    "n_train": len(X_train),
    "n_test": len(X_test),
}

print(f"\nAccuracy: {acc:.4f}")
print("\nClassification Report:")
print(report)
print("\nConfusion Matrix [humano, ia]:")
print(np.array(cm))


Accuracy: 0.9559

Classification Report:
              precision    recall  f1-score   support

      humano     1.0000    0.9109    0.9534       101
          ia     0.9196    1.0000    0.9581       103

    accuracy                         0.9559       204
   macro avg     0.9598    0.9554    0.9558       204
weighted avg     0.9594    0.9559    0.9558       204


Confusion Matrix [humano, ia]:
[[ 92   9]
 [  0 103]]


Assim, observa-se que o modelo BERT, apesar de ser exigente em termos de poder de computação, atinge um resultado excelente, com quase 96% de acurácia nos dados de teste. Além disso, obteve 100% de sucesso nos casos de textos humanos. Isso se dá pela capacidade do modelo de detectar o contexto das palavras e frases além de sua frequência, resultando em uma predição precisa.

# Cross-validation

Também foi implementado o cross-validation para os dois modelos a fim de perceber se algum deles varia muito conforme o conjunto de treino, o que indicaria um caso de *over-fitting* do modelo.




In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

# Load dataset
df = pd.read_json("https://github.com/Becker23/projeto-cd-ia/raw/refs/heads/main/dataset_final.json")
# Define pipeline
pipeline = Pipeline(
    [
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),  # unigrams + bigrams
                max_df=0.9,
                min_df=1,
                max_features=20000,
            ),
        ),
        ("clf", LinearSVC(C=1.0)),
    ]
)

# Perform 5-fold cross-validation
cv_scores = cross_val_score(
    pipeline, df["texto"], df["classe"], cv=5, scoring="accuracy"
)

# Print cross-validation results
print("\nCross-validation results:")
print(f"Mean accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
print(f"Individual fold scores: {cv_scores}")



Cross-validation results:
Mean accuracy: 0.7557 (+/- 0.1571)
Individual fold scores: [0.69117647 0.75735294 0.64705882 0.83088235 0.85185185]


Observa-se então que a acurácia média do modelo SVC é de aproximadamente 76%, tendo uma variação considerável entre os 5 casos (de 64% até 85%). Assim, isso indica que este modelo sofre de over-fitting, ou seja, um sobreajuste aos dados utilizados para treinamento/teste.

Além disso, o uso de conjuntos de palavras maiores (como trigramas, tetragramas, pentagramas, etc.) não indica uma clara melhoria na acurácia do modelo, como visto a seguir:

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

# Load dataset
df = pd.read_json("https://github.com/Becker23/projeto-cd-ia/raw/refs/heads/main/dataset_final.json")
for i in range(2, 6):
  pipeline = Pipeline(
      [
          (
              "tfidf",
              TfidfVectorizer(
                  lowercase=True,
                  ngram_range=(1, i),
                  max_df=0.9,
                  min_df=1,
                  max_features=20000,
              ),
          ),
          ("clf", LinearSVC(C=1.0)),
      ]
  )

# Perform 5-fold cross-validation
  cv_scores = cross_val_score(
    pipeline, df["texto"], df["classe"], cv=5, scoring="accuracy"
  )

# Print cross-validation results
  print(f"\nCross-validation results (i = {i}):")
  print(f"Mean accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
  print(f"Individual fold scores: {cv_scores}")



Cross-validation results (i = 2):
Mean accuracy: 0.7557 (+/- 0.1571)
Individual fold scores: [0.69117647 0.75735294 0.64705882 0.83088235 0.85185185]

Cross-validation results (i = 3):
Mean accuracy: 0.7571 (+/- 0.1635)
Individual fold scores: [0.68382353 0.76470588 0.64705882 0.83088235 0.85925926]

Cross-validation results (i = 4):
Mean accuracy: 0.7601 (+/- 0.1676)
Individual fold scores: [0.68382353 0.77205882 0.64705882 0.83088235 0.86666667]

Cross-validation results (i = 5):
Mean accuracy: 0.7616 (+/- 0.1764)
Individual fold scores: [0.68382353 0.75735294 0.64705882 0.84558824 0.87407407]


Sendo assim, o uso de conjuntos maiores de palavras não serve nenhum propósito neste caso.

Para o modelo BERT também foi implementado o cross-validation, apesar de neste caso ser um algoritmo exaustivo, pois envolve o treinamento do modelo com 5 épocas, 5 vezes diferentes. Sendo assim, o algoritmo tem 25 diferentes iterações de treino, o que pode tomar um tempo considerável a depender da máquina utilizada.

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd
import numpy as np
from pathlib import Path
import json

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MAX_LENGTH = 256
BATCH_SIZE = 4
EPOCHS = 5
LEARNING_RATE = 2e-5


# Custom dataset class
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx])
        label = 1 if self.labels.iloc[idx] == "ia" else 0

        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "labels": torch.tensor(label, dtype=torch.long),
        }

def train_model(model, train_loader, optimizer, device):
    model.train()
    total_loss = 0

    for batch_idx, batch in enumerate(train_loader):
        try:
            optimizer.zero_grad()
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids, attention_mask=attention_mask, labels=labels
            )

            loss = outputs.loss
            total_loss += loss.item()

            loss.backward()
            optimizer.step()

            # Explicitly clear some memory
            del outputs
            del loss
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        except RuntimeError as e:
            print(f"Error in batch {batch_idx}: {str(e)}")
            raise

    return total_loss


def evaluate_model(model, test_loader, device):
    model.eval()
    predictions = []
    actual_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"]

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            _, preds = torch.max(outputs.logits, dim=1)
            predictions.extend(preds.cpu().tolist())
            actual_labels.extend(labels.cpu().tolist())

    return predictions, actual_labels


def main():
    # Load data
    df = pd.read_json("https://github.com/Becker23/projeto-cd-ia/raw/refs/heads/main/dataset_final.json")

    # Split data
    from sklearn.model_selection import train_test_split

    print("\nPerforming 5-fold cross-validation...")
    from sklearn.model_selection import KFold

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(df), 1):
        print(f"\nFold {fold}/5")

        # Split data for this fold
        X_train = df["texto"].iloc[train_idx]
        y_train = df["classe"].iloc[train_idx]
        X_val = df["texto"].iloc[val_idx]
        y_val = df["classe"].iloc[val_idx]

        # Initialize model and tokenizer
        tokenizer = BertTokenizer.from_pretrained(
            "neuralmind/bert-base-portuguese-cased"
        )
        model = BertForSequenceClassification.from_pretrained(
            "neuralmind/bert-base-portuguese-cased", num_labels=2
        ).to(device)

        # Create datasets
        train_dataset = TextClassificationDataset(
            X_train, y_train, tokenizer, MAX_LENGTH
        )
        val_dataset = TextClassificationDataset(X_val, y_val, tokenizer, MAX_LENGTH)

        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

        # Train
        optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
        for epoch in range(EPOCHS):
            total_loss = train_model(model, train_loader, optimizer, device)
            print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")

        # Evaluate
        predictions, actual_labels = evaluate_model(model, val_loader, device)

        # Calculate accuracy
        label_map = {0: "humano", 1: "ia"}
        pred_labels = [label_map[p] for p in predictions]
        true_labels = [label_map[l] for l in actual_labels]
        fold_acc = accuracy_score(true_labels, pred_labels)

        cv_scores.append(fold_acc)
        print(f"Fold {fold} accuracy: {fold_acc:.4f}")

        # Clean up memory
        del model
        del tokenizer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Print cross-validation results
    cv_mean = np.mean(cv_scores)
    cv_std = np.std(cv_scores)
    print("\nCross-validation results:")
    print(f"Mean accuracy: {cv_mean:.4f} (+/- {cv_std * 2:.4f})")
    print(f"Individual fold scores: {cv_scores}")


if __name__ == "__main__":
    main()



Performing 5-fold cross-validation...

Fold 1/5


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/5, Loss: 0.4491
Epoch 2/5, Loss: 0.1117
Epoch 3/5, Loss: 0.1003
Epoch 4/5, Loss: 0.0350
Epoch 5/5, Loss: 0.0042
Fold 1 accuracy: 0.9338

Fold 2/5


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/5, Loss: 0.4156
Epoch 2/5, Loss: 0.1003
Epoch 3/5, Loss: 0.0270
Epoch 4/5, Loss: 0.0027
Epoch 5/5, Loss: 0.0706
Fold 2 accuracy: 0.9559

Fold 3/5


...

Epoch 1/5, Loss: 0.4977
Epoch 2/5, Loss: 0.1537
Epoch 3/5, Loss: 0.0573
Epoch 4/5, Loss: 0.0055
Epoch 5/5, Loss: 0.0131
Fold 3 accuracy: 0.9559

Fold 4/5


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/5, Loss: 0.4368
Epoch 2/5, Loss: 0.1106
Epoch 3/5, Loss: 0.0542
Epoch 4/5, Loss: 0.0059
Epoch 5/5, Loss: 0.0559
Fold 4 accuracy: 0.9044

Fold 5/5


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/5, Loss: 0.4280
Epoch 2/5, Loss: 0.0964
Epoch 3/5, Loss: 0.0551
Epoch 4/5, Loss: 0.0323
Epoch 5/5, Loss: 0.0514
Fold 5 accuracy: 0.9556

Cross-validation results:
Mean accuracy: 0.9411 (+/- 0.0404)
Individual fold scores: [0.9338235294117647, 0.9558823529411765, 0.9558823529411765, 0.9044117647058824, 0.9555555555555556]


A acurácia média do BERT foi de aproximadamente 94%, com variações de 90% até 96%. Sendo assim, podemos dizer que o modelo não sofreu de over-fitting, atingindo resultados muito parecidos com diferentes conjuntos de treinamento/teste.

# Resultados e conclusões

O modelo TF-IDF teve um bom desempenho geral, com aproximadamente 76% de acurácia média entre as 5 iterações. Ele funcionou especialmente bem com textos mais estruturados.
Porém, apresentou dificuldade em reconhecer nuances de escrita humana mais naturais ou emocionais, especialmente nos reviews, mesmo com ajustes nos conjuntos de palavras. Além disso, o modelo sofre de over-fitting, variando bastante em acurácia de acordo com os dados utilizados para treinamento/teste.

O modelo BERT superou o modelo tradicional.
Ele atingiu aproximadamente 94% de acurácia média.
Além disso, foi extremamente preciso em detectar textos gerados por IA.
Isso acontece pois o BERT é capaz de capturar padrões discursivos e não apenas o vocabulário. Por fim, é um modelo que se mostra robusto ao não sofrer de over-fitting, mostrando que não depende dos dados utilizados em treinamento/teste para sua precisão.

Vale lembrar também que até mesmo o hardware usado para o treino do modelo BERT pode interferir em sua acurácia. Sendo assim, é necessário ajustar os parâmetros dependendo da máquina utilizada no treinamento, a fim de obter resultados ainda melhores.


# Teste dos modelos

Abaixo é disponibilizado um algoritmo simples para teste dos modelos, onde um mesmo texto é analisado e classificado pelos dois modelos.

In [ ]:
# @title Insira texto para verificar a performance dos modelos
text = "Este texto não foi escrito por uma IA!" # @param {type:"string", placeholder:"Insira texto a ser analisado"}

# Prevendo com SVM

X_test_tfidf = vectorizer.transform([text])
y_pred = model_svc.predict(X_test_tfidf)

print(f"SVM: {y_pred[0]}")

# Prevendo com BERTimbau

encodings = tokenizer(
    [text],
    add_special_tokens=True,
    max_length=MAX_LENGTH,
    padding="max_length",
    truncation=True,
    return_tensors="pt"
)

input_ids = encodings["input_ids"].to(device)
attention_mask = encodings["attention_mask"].to(device)

with torch.no_grad():
    outputs = model_bert(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=1)

label_map = {0: "humano", 1: "ia"}
pred_labels = [label_map[p.item()] for p in predictions]

print(f"BERTimbau: {pred_labels[0]}")

SVM: ia
BERTimbau: humano
